# Diabetic retinopathy — process existing data in place
Set `RAW_DATASET_DIR` to the current folder containing `train`, `test`, `sample`, and `trainLabels.csv`. It can be on any local drive and does not need to be inside the project.

`PROJECT_ROOT` controls outputs. The three completed TFDS datasets are saved directly to:
- `PROJECT_ROOT/data/1M/`
- `PROJECT_ROOT/data/250K/`
- `PROJECT_ROOT/data/btgraham-300/`

No ZIPs are created. Raw images are read in place and never moved or copied. TFRecord shards, TFDS metadata and a build report sit directly in each output folder. Use `tfds.builder_from_directory` to read these portable folders.

TFDS builds under `data/_tfds_build` and the completed version directory is renamed into its final folder on the same drive (no second dataset copy). Small smoke-test datasets and the optional published test-label download use `data/_tfds_aux`. Completed final outputs are reused on rerun; interrupted builds may restart.

The first cell writes `project_paths.py` for importing `PROJECT_ROOT` in later scripts run from the project root. Your current environment's packages are retained. Disconnected mode is the default; provide `retinopathy_solution.csv` in the raw folder, or enable its one-time download below.

In [5]:
import sys
import tensorflow as tf
import tensorflow_datasets as tfds

print("Python:", sys.executable)
print("TensorFlow:", tf.__version__)
print("TFDS:", tfds.__version__)

Python: C:\Users\kawal\miniconda3\envs\dr_tf\python.exe
TensorFlow: 2.10.1
TFDS: 4.8.3


In [6]:
from pathlib import Path
import sys, os, json, csv, shutil, hashlib
from collections import Counter

PROJECT_ROOT = Path(r'D:\M L\Diabetic_Retinopathy_Detection_mini_project')  # path here
RAW_DATASET_DIR = Path(r'data')  # path of the folder shown in your screenshot
RAW = RAW_DATASET_DIR
MANUAL = RAW  # TFDS manual source; input locations are resolved below
DATA = PROJECT_ROOT / 'data'
BUILD = DATA / '_tfds_build'
AUX = DATA / '_tfds_aux'
CONFIGS = ('1M', '250K', 'btgraham-300')
ALLOW_LABEL_DOWNLOAD = False
assert RAW.is_dir(), f'Set RAW_DATASET_DIR to your existing extracted folder: {RAW}'
for folder in (DATA, BUILD, AUX):
    folder.mkdir(parents=True, exist_ok=True)

# A shared variable for later scripts, not another hard-coded path in every script.
config_file = PROJECT_ROOT / 'project_paths.py'
config_text = 'from pathlib import Path\n\nPROJECT_ROOT = Path(__file__).resolve().parent\n'
if config_file.exists():
    assert config_file.read_text() == config_text, f'Existing {config_file} differs; review it before continuing.'
else:
    config_file.write_text(config_text)
print('Interpreter:', sys.executable)
print('Python:', sys.version)
print('Project:', PROJECT_ROOT)
os.environ['TFDS_DISABLE_GCS'] = '1'
import tensorflow as tf
import tensorflow_datasets as tfds
import numpy as np
import cv2
VERSIONS = dict(tensorflow=tf.__version__, tfds=tfds.__version__,
                numpy=np.__version__, opencv=cv2.__version__, python=sys.version)
print(json.dumps(VERSIONS, indent=2))

Interpreter: C:\Users\kawal\miniconda3\envs\dr_tf\python.exe
Python: 3.10.20 | packaged by Anaconda, Inc. | (main, Jun 11 2026, 15:13:20) [MSC v.1942 64 bit (AMD64)]
Project: D:\M L\Diabetic_Retinopathy_Detection_mini_project
{
  "tensorflow": "2.10.1",
  "tfds": "4.8.3",
  "numpy": "1.23.5",
  "opencv": "4.8.1",
  "python": "3.10.20 | packaged by Anaconda, Inc. | (main, Jun 11 2026, 15:13:20) [MSC v.1942 64 bit (AMD64)]"
}


Check that the interpreter above belongs to your project environment. Existing package versions are retained. The next cell verifies folder structure and image counts without changing your data. It accepts either images directly in `RAW_DATASET_DIR/train`, or a nested folder such as `RAW_DATASET_DIR/train/train`; ambiguous multiple image directories stop with a clear error.

In [7]:
def need_space(folder, amount):
    free = shutil.disk_usage(folder).free
    if free < amount:
        raise RuntimeError(f'{folder}: need {amount/2**30:.1f} GiB free, have {free/2**30:.1f}')

def image_folder(name, expected):
    base = RAW / name
    assert base.is_dir(), f'Missing folder: {base}'
    images = list(base.rglob('*.jpeg'))
    parents = {p.parent for p in images}
    assert len(parents) == 1, f'{name}: expected one image directory, found {parents}'
    assert len(images) == expected, f'{name}: expected {expected:,} JPEGs, found {len(images):,}'
    folder = parents.pop()
    print(name, f'{len(images):,}', folder)
    return folder

IMAGE_DIRS = {name: image_folder(name, count) for name, count in
              [('train',35126), ('test',53576), ('sample',10)]}

def csv_file(name, optional=False):
    candidate = RAW / name
    if candidate.is_file():
        return candidate
    matches = [p for p in candidate.rglob(name) if p.is_file()] if candidate.is_dir() else []
    if len(matches) == 1:
        return matches[0]
    if optional and not matches:
        return None
    raise FileNotFoundError(f'Expected one actual {name} file at or inside {candidate}; found {matches}')

TRAIN_LABELS = csv_file('trainLabels.csv')
print('Actual training CSV:', TRAIN_LABELS)
print('Free output disk:', round(shutil.disk_usage(DATA).free / 2**30, 1), 'GiB')

train 35,126 data\train\train\train
test 53,576 data\test\test\test
sample 10 data\sample
Actual training CSV: data\trainLabels.csv\trainLabels.csv
Free output disk: 94.8 GiB


Load the published validation/test CSV and verify every labeled filename against the images. The adapter redirects TFDS to your existing image/CSV locations and local test-label file. Native image preprocessing and Public/Private splitting are preserved. A separate two-example-per-split smoke build checks all three configurations before the full workload.

In [8]:
LABEL_URL = 'https://storage.googleapis.com/kaggle-forum-message-attachments/90528/2877/retinopathy_solution.csv'
solution = csv_file('retinopathy_solution.csv', optional=True) or (AUX / 'retinopathy_solution.csv')
if not solution.exists():
    if not ALLOW_LABEL_DOWNLOAD:
        raise FileNotFoundError(f'Place retinopathy_solution.csv in {RAW}, or set ALLOW_LABEL_DOWNLOAD=True once.')
    import urllib.request
    partial = solution.with_suffix('.csv.partial')
    with urllib.request.urlopen(LABEL_URL, timeout=60) as src, partial.open('wb') as dst:
        shutil.copyfileobj(src, dst)
    # Validate before promoting a response to a completed label file.
    with partial.open(newline='', encoding='utf-8-sig') as f:
        check = list(csv.DictReader(f))
    assert len(check) == 53576 and {'image','level','Usage'} <= set(check[0])
    partial.replace(solution)

rows_by_split = {}
for split, file, total in [('train', TRAIN_LABELS,35126), ('test', solution,53576)]:
    with file.open(newline='', encoding='utf-8-sig') as f:
        rows = list(csv.DictReader(f))
    assert len(rows) == total and len({r['image'] for r in rows}) == total
    assert all(int(r['level']) in range(5) for r in rows)
    expected_names = {r['image'] + '.jpeg' for r in rows}
    actual_names = {p.name for p in IMAGE_DIRS[split].glob('*.jpeg')}
    assert actual_names == expected_names, f'Missing/extra {split} images'
    rows_by_split[split] = rows
    print(split, 'class counts:', dict(sorted(Counter(int(r['level']) for r in rows).items())))
assert Counter(r['Usage'] for r in rows_by_split['test']) == {'Public':10906, 'Private':42670}

class LocalLabelsManager:
    def __init__(self, original):
        self.original = original
    def __getattr__(self, name):
        return getattr(self.original, name)
    def download(self, url):
        if url != LABEL_URL:
            raise RuntimeError(f'Unexpected download requested: {url}')
        return str(solution)

def local_builder(config, root):
    BuilderClass = tfds.builder_cls("diabetic_retinopathy_detection")
    builder = BuilderClass(
        config=config,
        version="3.0.0",
        data_dir=str(root),
    )
    original_splits = builder._split_generators
    def resolved_splits(dl_manager):
        splits = original_splits(LocalLabelsManager(dl_manager))
        # Preserve TFDS's Public/Private split rules; only adapt source locations.
        for split in splits:
            name = str(split.name)
            source = 'test' if name in ('test', 'validation') else name
            split.gen_kwargs['images_dir_path'] = str(IMAGE_DIRS[source])
            if name == 'train':
                split.gen_kwargs['csv_path'] = str(TRAIN_LABELS)
        return splits
    builder._split_generators = resolved_splits
    return builder

DOWNLOAD_CONFIG = tfds.download.DownloadConfig(manual_dir=str(MANUAL), try_download_gcs=False)
# Small end-to-end test in a separate directory: catches installed-version incompatibilities.
for config in CONFIGS:
    trial = local_builder(config, AUX/'smoke')
    trial.download_and_prepare(download_dir=str(AUX/'downloads'), download_config=tfds.download.DownloadConfig(
        manual_dir=str(MANUAL), try_download_gcs=False, max_examples_per_split=2), file_format='tfrecord')
    example = next(iter(trial.as_dataset(split='train').take(1)))
    print(config, example['image'].shape, int(example['label']))

train class counts: {0: 25810, 1: 2443, 2: 5292, 3: 873, 4: 708}
test class counts: {0: 39533, 1: 3762, 2: 7861, 3: 1214, 4: 1206}
1M (816, 1225, 3) 0
250K (408, 612, 3) 0
btgraham-300 (540, 540, 3) 0


Build each configuration sequentially. Read one decoded record from each split and check split counts before finalizing. The finished TFDS version folder is renamed directly to `data/<config>`; raw inputs stay in their current location. Existing final output folders are loaded and validated rather than overwritten.

In [9]:
EXPECTED_SPLITS = {'train':35126, 'validation':10906, 'test':42670, 'sample':10}
for config in CONFIGS:
    output = DATA / config
    if output.exists():
        builder = tfds.builder_from_directory(str(output))
        assert builder.info.name == 'diabetic_retinopathy_detection'
        assert str(builder.info.version) == '3.0.0'
        report = json.loads((output/'local_build.json').read_text())
        assert report['config'] == config, f'Unexpected config in {output}'
        print('Checking existing output:', output)
    else:
        need_space(DATA, 8*2**30)  # Conservative per-config build/shuffle headroom
        builder = local_builder(config, BUILD)
        builder.download_and_prepare(download_dir=str(AUX/'downloads'),
                                     download_config=DOWNLOAD_CONFIG, file_format='tfrecord')
    counts = {k:v.num_examples for k,v in builder.info.splits.items()}
    assert counts == EXPECTED_SPLITS, counts
    for split in counts:
        record = next(iter(builder.as_dataset(split=split).take(1)))
        assert record['image'].shape[-1] == 3
    if not output.exists():
        folder = Path(builder.data_dir)
        build_info = dict(versions=VERSIONS, splits=counts, config=config,
                          raw_dataset_dir=str(RAW_DATASET_DIR),
                          test_labels_sha256=hashlib.sha256(solution.read_bytes()).hexdigest())
        (folder/'local_build.json').write_text(json.dumps(build_info, indent=2))
        # Release dataset iterators above before renaming; output is on the same drive.
        folder.rename(output)
    print('Ready:', output)
print('All three processed datasets are available directly under', DATA)


Checking existing output: D:\M L\Diabetic_Retinopathy_Detection_mini_project\data\1M
Ready: D:\M L\Diabetic_Retinopathy_Detection_mini_project\data\1M
Checking existing output: D:\M L\Diabetic_Retinopathy_Detection_mini_project\data\250K
Ready: D:\M L\Diabetic_Retinopathy_Detection_mini_project\data\250K
Checking existing output: D:\M L\Diabetic_Retinopathy_Detection_mini_project\data\btgraham-300
Ready: D:\M L\Diabetic_Retinopathy_Detection_mini_project\data\btgraham-300
All three processed datasets are available directly under D:\M L\Diabetic_Retinopathy_Detection_mini_project\data


In [10]:
# Read directly from the processing-type folder (no extraction required).
builder = tfds.builder_from_directory(str(PROJECT_ROOT / 'data' / '250K'))
train_ds = builder.as_dataset(split='train', shuffle_files=True)
print(train_ds.element_spec)
# Each example is a dict: image, label, name. Image dimensions vary.


{'image': TensorSpec(shape=(None, None, 3), dtype=tf.uint8, name=None), 'label': TensorSpec(shape=(), dtype=tf.int64, name=None), 'name': TensorSpec(shape=(), dtype=tf.string, name=None)}


Sources: [TFDS builder](https://github.com/tensorflow/datasets/blob/master/tensorflow_datasets/image_classification/diabetic_retinopathy_detection.py), [catalog](https://www.tensorflow.org/datasets/catalog/diabetic_retinopathy_detection), [DownloadConfig](https://www.tensorflow.org/datasets/api_docs/python/tfds/download/DownloadConfig).

The local-label adapter uses the builder's private `_split_generators` hook to redirect its published CSV download and resolve existing input paths. The separate smoke build tests this against your installed TFDS version before the full workload. No image processing algorithm has been reimplemented. The notebook was syntax and notebook structure checked in the authoring workspace; the actual Kaggle dataset and your local environment were not available there.